In [2]:
import sys
sys.path.insert(0, '/app')

from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, lit, sha2, struct, to_json, current_timestamp
from config.settings import oracle_config, clickhouse_config, spark_config

ref_date = datetime.now().strftime('%Y-%m-%d')
table_name = 'YOUR_SOURCE_TABLE'
schema_name = 'ADMBI_PRD'
primary_keys = ['ID']
arrow_hash_fields = ['COLUMN_1', 'COLUMN_2']

master_url = 'spark://spark-master:7077'
spark = (
    SparkSession.builder
    .appName(f'BronzeIngestion_{table_name}_{ref_date}')
    .config('spark.master', master_url)
    .config('spark.driver.memory', spark_config.driver_memory)
    .config('spark.executor.memory', spark_config.executor_memory)
    .config('spark.executor.cores', spark_config.executor_cores)
    .config('spark.sql.shuffle.partitions', spark_config.sql_shuffle_partitions)
    .config('spark.sql.adaptive.enabled', spark_config.sql_adaptive_enabled)
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .getOrCreate()
)



ModuleNotFoundError: No module named 'pyspark'

In [15]:
oracle_jdbc_url = f"jdbc:oracle:thin:@//{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"

source_df = (
    spark.read
    .format('jdbc')
    .option('url', oracle_jdbc_url)
    .option('dbtable', f"{schema_name}.{table_name}")
    .option('user', oracle_config.user)
    .option('password', oracle_config.password)
    .option('driver', 'oracle.jdbc.driver.OracleDriver')
    .option('fetchsize', '10000')
    .option('numPartitions', '10')
    .load()
)

source_df.printSchema()

primary_key_expr = concat_ws('|||', *[col(pk) for pk in primary_keys])
row_hash_expr = sha2(concat_ws('|||', *[col(c.name) for c in source_df.schema]), 256)
json_expr = to_json(struct(*[col(c.name) for c in source_df.schema]))

bronze_df = (
    source_df
    .select(
        lit(ref_date).cast('date').alias('ref_date'),
        lit(f"{schema_name}.{table_name}").alias('table_name'),
        primary_key_expr.alias('primary_key'),
        row_hash_expr.alias('row_hash'),
        json_expr.alias('data'),
        current_timestamp().alias('ingestion_timestamp')
    )
)

(
    bronze_df.write
    .format('jdbc')
    .option('url', f"jdbc:clickhouse://{clickhouse_config.host}:{clickhouse_config.port}/{clickhouse_config.database}")
    .option('dbtable', 'bronze.snapshot_raw')
    .option('user', clickhouse_config.user)
    .option('password', clickhouse_config.password)
    .option('driver', 'com.clickhouse.jdbc.ClickHouseDriver')
    .option('batchsize', '500000')
    .mode('append')
    .save()
)



Py4JJavaError: An error occurred while calling o78.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=sg9u+1kjTDKIJXMQuidhsQ==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.sca

In [6]:
clickhouse_jdbc_url = f"jdbc:clickhouse://{clickhouse_config.host}:{clickhouse_config.port}/{clickhouse_config.database}"

(
    bronze_df.write
    .format('jdbc')
    .option('url', clickhouse_jdbc_url)
    .option('dbtable', 'bronze.snapshot_raw')
    .option('user', clickhouse_config.user)
    .option('password', clickhouse_config.password)
    .option('driver', 'com.clickhouse.jdbc.ClickHouseDriver')
    .option('batchsize', '500000')
    .mode('append')
    .save()
)

spark.stop()



NameError: name 'bronze_df' is not defined

In [ ]:
import sys
sys.path.insert(0, '/app')

from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat_ws, sha2, to_json, struct, current_timestamp,
)
from config.settings import oracle_config, clickhouse_config, spark_config, app_config

In [9]:
ref_date = datetime.now().strftime("%Y-%m-%d")
ref_date


'2025-12-05'

In [ ]:
import os
spark_home = os.environ.get('SPARK_HOME', '/usr/local/spark')
ojdbc_jar = f"{spark_home}/jars/ojdbc8-21.9.0.0.jar"
clickhouse_jar = f"{spark_home}/jars/clickhouse-jdbc-0.4.6-all.jar"

spark = (
    SparkSession.builder
    .appName(f"BronzeIngestion_{ref_date}")
    .config("spark.driver.memory", spark_config.driver_memory)
    .config("spark.executor.memory", spark_config.executor_memory)
    .config("spark.executor.cores", spark_config.executor_cores)
    .config("spark.sql.shuffle.partitions", spark_config.sql_shuffle_partitions)
    .config("spark.sql.adaptive.enabled", spark_config.sql_adaptive_enabled)
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.jars", f"{ojdbc_jar},{clickhouse_jar}")
    .getOrCreate()
)


In [ ]:
oracle_jdbc_url = f"jdbc:oracle:thin:@//{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"
oracle_jdbc_url


'jdbc:oracle:thin:@10.255.150.11:1521/bi.grupotracker.com.br'

In [12]:
schema_name = "ADMBI_PRD"
table_name = "YOUR_TABLE_NAME"
primary_keys = ["ID"]


In [13]:
df_oracle = (
    spark.read
    .format("jdbc")
    .option("url", oracle_jdbc_url)
    .option("dbtable", f"{schema_name}.{table_name}")
    .option("user", oracle_config.user)
    .option("password", oracle_config.password)
    .option("driver", "oracle.jdbc.driver.OracleDriver")
    .option("fetchsize", app_config.batch_size)
    .option("numPartitions", "10")
    .load()
)


Py4JJavaError: An error occurred while calling o63.load.
: java.sql.SQLRecoverableException: IO Error: The Network Adapter could not establish the connection (CONNECTION_ID=YEBvr0TAQIanMLFvthStBw==)
	at oracle.jdbc.driver.T4CConnection.handleLogonNetException(T4CConnection.java:892)
	at oracle.jdbc.driver.T4CConnection.logon(T4CConnection.java:697)
	at oracle.jdbc.driver.PhysicalConnection.connect(PhysicalConnection.java:1047)
	at oracle.jdbc.driver.T4CDriverExtension.getConnection(T4CDriverExtension.java:89)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:732)
	at oracle.jdbc.driver.OracleDriver.connect(OracleDriver.java:648)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.sca

In [ ]:
df_oracle.printSchema()


In [ ]:
primary_key_str = concat_ws("|||", *[col(pk) for pk in primary_keys])
all_columns = [col(c) for c in df_oracle.columns]
row_data = to_json(struct(*all_columns))
row_hash_input = concat_ws("|||", *all_columns)


In [ ]:
df_bronze = df_oracle.select(
    lit(ref_date).cast("date").alias("ref_date"),
    lit(table_name).alias("table_name"),
    primary_key_str.alias("primary_key"),
    sha2(row_hash_input, 256).alias("row_hash"),
    row_data.alias("data"),
    current_timestamp().alias("ingestion_timestamp"),
)


In [ ]:
df_bronze.show(5, truncate=False)


In [ ]:
protocol = "https" if clickhouse_config.secure else "http"
clickhouse_jdbc_url = (
    f"jdbc:clickhouse://{protocol}://{clickhouse_config.host}:"
    f"{clickhouse_config.port}/{clickhouse_config.database}"
)
clickhouse_jdbc_url


In [ ]:
df_bronze.write \
    .format("jdbc") \
    .option("url", clickhouse_jdbc_url) \
    .option("dbtable", "bronze.snapshot_raw") \
    .option("user", clickhouse_config.user) \
    .option("password", clickhouse_config.password) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .option("batchsize", app_config.batch_size) \
    .mode("append") \
    .save()


In [ ]:
spark.stop()
